# DIAL ALERT reproducible workflow

This notebook reproduces the capstone pipeline in auditable stages. DIAL-ALERT is an academic prototype and is not intended for clinical care.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
print(ROOT)

## 1 Obtain the fixed public dataset

The acquisition script downloads HEMOBP Version 3 and verifies the publisher-provided MD5 checksums. The largest file is about 251 MiB.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'src/download_data.py')], cwd=ROOT, check=True)

## 2 Build the leakage-controlled session table

The index observation is the earliest valid active-dialysis reading within minutes 0 to 30. Outcomes use later readings only, and history variables use earlier sessions only.

In [ ]:
subprocess.run([
    sys.executable, str(ROOT / 'src/build_session_dataset.py'),
    '--raw-dir', str(ROOT / 'data/raw'),
    '--output-dir', str(ROOT / 'data/processed'),
], cwd=ROOT, check=True)

In [ ]:
flow = json.loads((ROOT / 'data/processed/cohort_flow.json').read_text())
pd.Series(flow, name='value').to_frame()

## 3 Train and compare models

This stage uses patient-grouped cross-validation and preserves patient-disjoint training, validation, and test partitions. It may take several minutes.

In [ ]:
subprocess.run([
    sys.executable, str(ROOT / 'src/train_evaluate.py'),
    '--data', str(ROOT / 'data/processed/hemobp_session_level.csv.gz'),
    '--config', str(ROOT / 'configs/model_config.json'),
    '--artifacts', str(ROOT / 'artifacts'),
    '--models', str(ROOT / 'models'),
], cwd=ROOT, check=True)

In [ ]:
comparison = pd.read_csv(ROOT / 'artifacts/model_comparison.csv')
comparison[[
    'model', 'cv_average_precision_mean',
    'validation_average_precision', 'validation_brier_score'
]].sort_values('cv_average_precision_mean', ascending=False)

## 4 Run the ethical AI and bias audit

The audit reports explainability, uncertainty, subgroup metrics, intersections, and mitigation experiments. It does not establish causal fairness.

In [ ]:
subprocess.run([sys.executable, str(ROOT / 'src/audit_bias_fairness.py')], cwd=ROOT, check=True)

In [ ]:
test_result = json.loads((ROOT / 'artifacts/final_test_metrics.json').read_text())
pd.Series(test_result['test_metrics_calibrated'], name='test_value').to_frame()

## 5 Verify repository contracts

The tests check feature ordering, result consistency, publication files, and saved-model inference.

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q'], cwd=ROOT, check=True)